<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.4-ai-chat-with-fastapi/notebooks/exercises-5.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.4 — AI Chat with FastAPI
Netsetos GenAI Course v2.0 | Module 5

Production chat API: sessions, streaming, deployment, testing, India


In [ ]:
# pip install fastapi uvicorn openai pydantic-settings httpx -q
print('Ready to build AI chat API')


## Ex 1: Basic Chat Endpoint


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal, Optional

app = FastAPI(title='AI Chat API', version='1.0.0')

class ChatMessage(BaseModel):
    role: Literal['system', 'user', 'assistant']
    content: str = Field(..., min_length=1, max_length=32000)

class ChatRequest(BaseModel):
    messages: list[ChatMessage] = Field(..., min_length=1)
    model: str = Field(default='gpt-4o-mini')
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    stream: bool = Field(default=False)

class ChatResponse(BaseModel):
    content: str
    model: str
    usage: dict

@app.post('/chat', response_model=ChatResponse)
async def chat(request: ChatRequest):
    # In production: call AsyncOpenAI here
    return ChatResponse(
        content='Hello! This is a mock response.',
        model=request.model,
        usage={'prompt_tokens': 10, 'completion_tokens': 20}
    )

print('Chat endpoint defined. Run: uvicorn main:app --reload')


## Ex 2: Session Management


In [ ]:
from uuid import uuid4, UUID
from datetime import datetime

conversations: dict[str, list] = {}

def create_session() -> str:
    session_id = str(uuid4())
    conversations[session_id] = []
    return session_id

def add_message(session_id: str, role: str, content: str):
    conversations[session_id].append({
        'role': role, 'content': content,
        'timestamp': datetime.now().isoformat()
    })

def get_history(session_id: str) -> list:
    return conversations.get(session_id, [])

# Demo
sid = create_session()
add_message(sid, 'user', 'Hello!')
add_message(sid, 'assistant', 'Hi! How can I help?')
add_message(sid, 'user', 'What is FastAPI?')
print(f'Session {sid[:8]}... has {len(get_history(sid))} messages')


## Ex 3: Context Window Strategy


In [ ]:
import tiktoken

def count_tokens(messages: list, model='gpt-4o') -> int:
    enc = tiktoken.encoding_for_model(model)
    return sum(len(enc.encode(m['content'])) + 4 for m in messages)

def sliding_window(messages: list, max_tokens: int = 4000, model='gpt-4o') -> list:
    """Keep most recent messages within token budget"""
    result = []
    total = 0
    for msg in reversed(messages):
        tokens = count_tokens([msg], model)
        if total + tokens > max_tokens:
            break
        result.insert(0, msg)
        total += tokens
    return result

# Demo: 50-turn conversation cost
messages = [{'role': 'user' if i%2==0 else 'assistant',
             'content': 'A' * 500} for i in range(50)]
full_tokens = count_tokens(messages)
windowed = sliding_window(messages, max_tokens=4000)
window_tokens = count_tokens(windowed)
print(f'Full history: {full_tokens} tokens (${full_tokens * 2.50 / 1e6:.4f})')
print(f'Windowed (last {len(windowed)} msgs): {window_tokens} tokens (${window_tokens * 2.50 / 1e6:.4f})')
print(f'Savings: {(1 - window_tokens/full_tokens)*100:.0f}%')


## Ex 4: SSE Streaming Pattern


In [ ]:
import asyncio, json
from fastapi.responses import StreamingResponse

async def stream_tokens(messages: list):
    """Async generator for SSE streaming"""
    # Simulated token streaming
    response_text = 'FastAPI makes building AI chat APIs straightforward.'
    for word in response_text.split():
        yield f'data: {json.dumps({"content": word + " "})}\n\n'
        await asyncio.sleep(0.1)  # Simulate token generation
    yield 'data: [DONE]\n\n'

# Usage in endpoint:
# return StreamingResponse(
#     stream_tokens(messages),
#     media_type='text/event-stream',
#     headers={'Cache-Control': 'no-cache', 'X-Accel-Buffering': 'no'}
# )
print('SSE streaming pattern ready')
print('Headers needed: Cache-Control: no-cache, X-Accel-Buffering: no')


## Ex 5: Dependency Injection


In [ ]:
from typing import Annotated

# Type aliases for clean endpoints
# DbSession = Annotated[AsyncSession, Depends(get_db)]
# CurrentUser = Annotated[dict, Depends(verify_token)]
# LLMClient = Annotated[AsyncOpenAI, Depends(get_llm_client)]

print('Dependency Injection Patterns:')
print()
patterns = [
    ('Database', 'Async generator, auto-commit/rollback', 'get_db()'),
    ('LLM Client', 'Lifespan singleton on app.state', 'get_llm_client()'),
    ('Auth', 'JWT verification dependency', 'verify_token()'),
    ('Rate Limiter', 'Redis-backed token counter', 'check_rate_limit()'),
]
for name, desc, func in patterns:
    print(f'  {name:15s}: {func:25s} — {desc}')


## Ex 6: Error Handling


In [ ]:
print('Error Handling Architecture:')
print()
errors = [
    ('LLMRateLimitError', 429, 'Retry-After header', 'Provider rate limit hit'),
    ('LLMTimeoutError', 504, 'None', 'LLM response took >30s'),
    ('ContextTooLongError', 400, 'max_tokens in body', 'Input exceeds context window'),
    ('ProviderDownError', 503, 'Retry-After: 60', 'All providers unavailable'),
    ('InvalidMessageError', 422, 'Validation details', 'Pydantic validation failed'),
]
print(f'{"Exception":25s} {"HTTP":>5s} {"Header":20s} {"When"}')
print('-'*80)
for exc, code, header, when in errors:
    print(f'{exc:25s} {code:>5d} {header:20s} {when}')


## Ex 7: Docker + Deployment


In [ ]:
dockerfile = '''# Multi-stage Dockerfile for FastAPI AI Chat
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /usr/local/lib/python3.12 /usr/local/lib/python3.12
COPY --from=builder /usr/local/bin /usr/local/bin
COPY . .

# Exec-form CMD (not shell form!)
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]
'''
print(dockerfile)
print('Deploy: gcloud run deploy ai-chat \\')
print('  --source . --region asia-south1 \\')
print('  --set-secrets="OPENAI_API_KEY=openai-key:latest" \\')
print('  --min-instances 1 --timeout 600')


## Ex 8: Testing Pattern


In [ ]:
print('Testing FastAPI Chat with pytest:')
print()
test_code = '''
import pytest
from unittest.mock import AsyncMock, MagicMock, patch
from httpx import AsyncClient, ASGITransport

def create_mock_response(content="Hello!"):
    mock = MagicMock()
    mock.choices = [MagicMock()]
    mock.choices[0].message.content = content
    mock.usage = MagicMock(prompt_tokens=10, completion_tokens=20)
    return mock

@pytest.mark.anyio
@patch("app.chat.service.openai_client.chat.completions.create", new_callable=AsyncMock)
async def test_chat_endpoint(mock_create):
    mock_create.return_value = create_mock_response("Hi!")
    transport = ASGITransport(app=app)
    async with AsyncClient(transport=transport, base_url="http://test") as client:
        response = await client.post("/chat", json={"messages": [{"role": "user", "content": "Hello"}]})
        assert response.status_code == 200
        assert "Hi!" in response.json()["content"]
'''
print(test_code)


---
## Recap
AsyncOpenAI (never sync in async routes — 97% RPS drop). Domain module structure. Dependency injection (DB sessions, LLM clients, auth). SSE streaming with StreamingResponse + X-Accel-Buffering. Context window strategies (sliding window, summarization). structlog + Langfuse observability. Docker multi-stage + Cloud Run. Rate limiting + OWASP prompt injection defense. pytest with mocked LLM. Sarvam AI drop-in for India (api.sarvam.ai/v1).
